# Tutorial

In [2]:
import plotly.io as pio
pio.renderers.default = "notebook_connected" # "notebook" or "notebook_connected" for offline use

*oddSNP* is an open-source framework for calculating SNP-Information Content (SNP-IC), a quantitative metric derived from unpooled pilot single-cell RNA sequencing (scRNA-seq) data that accurately predicts the success of genotype-based demultiplexing, as well as its counterpart for genotype-free approaches, cell-paired SNP-Information Content (cpSNP-IC).

Details on these metrics and the tool's implementation are available in the manuscript entitled: *OddSNP: a predictive framework for optimizing multiplexed single-cell RNA-seq experiments*.

## Installation

The recommended way to install *oddSNP* is by using a virtual-environment manager such as [Conda](https://docs.conda.io/projects/conda/en/stable/user-guide/install/index.html). Even when installing from source, we will create the corresponding environments prior to installation:

### Using Bioconda:

We create a new conda environment and directly install *oddSNP* from its bioconda source.
 
```bash
:~$ conda create --name oddsnp
:~$ conda activate oddsnp
(oddsnp):~$ conda install -c bioconda oddsnp
(oddsnp):~$
```

### Using PyPI:

Still, we recommend to install *oddSNP* inside a virtual environment. In this case, we need to make sure to also install `pip` to the created environment to avoid interfering with system libraries.

```bash
:~$ conda create --name oddsnp
:~$ conda activate oddsnp
(oddsnp):~$ conda install pip
(oddsnp):~$ pip install oddsnp
```

### From source

It is also possible to install *oddSNP* directly from source to a previously created environment.

For this, we first clone the contents of the GitHub repository to local folder (i.e `path/to/oddsnp`), then, in order to install *oddSNP* as a command line tool in the current environment, we simply use the following:

```bash
:~$ conda create --name oddsnp
:~$ conda activate oddsnp
(oddsnp):~$ conda install pip
(oddsnp):~$ cd path/to/oddsnp/
(oddsnp):~/path/to/oddsnp$ pip install .
```

In this case, make sure that the `pip` version used for installation is the one associated to the environment and not a system version (i.e. `$ which pip` should point to an environment directory and not your system's pip).

Notice that by running this command, you will also install to the currently activated environment all of the tool's dependencies.

### After installation

An installation of *[cellsnp-lite](https://cellsnp-lite.readthedocs.io/en/latest/index.html)* is required to perform pileup calculations within oddSNP. To install it, use the following command inside your activated conda environment:

```bash
(oddsnp):~$ conda install -c bioconda cellsnp-lite 
```

**NOTE** Other installation methods for `cellsnp-lite` are described in their [original website](https://cellsnp-lite.readthedocs.io/en/latest/index.html).

To check the installation finished properly, we can try and run *oddSNP* from the command line without any sub-commands. The output should be as follows:

```bash
(oddsnp):~$ oddSNP 
Usage: oddSNP [OPTIONS] COMMAND [ARGS]...

Options:
  --help  Show this message and exit.

Commands:
  cpsnpic
  downsample
  genotype
  snpic
  utils
```

**NOTE** The rest of this tutorial will use *oddSNP* through its available API. Notice that all functionality is also available through the corresponding combination of command-line instructions.

## Using oddSNP

### 1. Sample data preparation

First, we check the current working directory

In [2]:
import os

current_directory = os.getcwd()
print(f"Current working directory: {current_directory}")

Current working directory: /mnt/storage4/rallendes/oddSNP/notebooks


The minimum starting point for a *snpIC* workflow requires the following input files:
- a scRNAseq BAM file (together with a list of cell barcodes), and
- a SNP reference file, typically in VCF format (with or without minor Allelle frequency information)

Due to file size, we have not included the sample files used for this tutorial, however, they are publicly available for download from: [1k PBMCs from a Healthy Donor (v3 chemistry) BAM from 10XGenomics](https://www.10xgenomics.com/datasets/1-k-pbm-cs-from-a-healthy-donor-v-3-chemistry-3-standard-3-0-0); and [Call set from 1000 Genomes Project sequence against GRCh38 (SNV and indels)](https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/data_collections/1000_genomes_project/release/20190312_biallelic_SNV_and_INDEL/ALL.wgs.shapeit2_integrated_snvindels_v2a.GRCh38.27022019.sites.vcf.gz) respectively.

The following commands will automatically download and place the files in the required folders. Alternatively, you download the files manually and place them in the *{current_directory}/sample_data/* folder.

For downloading the bam file (~5GB) and its corresponding matrix file:

In [5]:
import os
import requests
import tarfile

# create a sample_data directory if it doesn't exist
if not os.path.exists(f"{current_directory}/sample_data"):
    os.makedirs(f"{current_directory}/sample_data")

# bam file
url = "https://cf.10xgenomics.com/samples/cell-exp/3.0.0/pbmc_1k_v3/pbmc_1k_v3_possorted_genome_bam.bam"
save_path = f"{current_directory}/sample_data/pbmc_1k_v3_possorted_genome_bam.bam"
response = requests.get(url)
with open(save_path, "wb") as f:
  f.write(response.content)

# matrix file (we use the list of barcodes from the filtered matrix generated by Cell Ranger))
url = "https://cf.10xgenomics.com/samples/cell-exp/3.0.0/pbmc_1k_v3/pbmc_1k_v3_filtered_feature_bc_matrix.tar.gz"
save_path = f"{current_directory}/sample_data/pbmc_1k_v3_filtered_feature_bc_matrix.tar.gz"
response = requests.get(url)
with open(save_path, "wb") as f:
  f.write(response.content)

# untar the matrix file
with tarfile.open(save_path, 'r:gz') as file:
  file.extractall(f"{current_directory}/sample_data/", filter='data') 
  file.close()

<div style="border-left: 5px solid #f39c12; padding: 10px; background-color: #fff3cd;">
<strong>⚠️ Warning:</strong> 
When using the 10x Chromium platform and Cell Ranger software, if the number of cell-associated barcodes reported in filtered_feature_bc_matrix greatly exceeds the intended target recovered cell count, review the result carefully before using the filtered barcode set for this analysis. Such an excess may indicate that automatic cell calling by cellranger count has retained low-confidence barcodes that may be classified as non-demultiplexable by demultiplexing algorithms, thereby affecting downstream analyses such as SNP-IC threshold-based prediction. Assess this possibility using the barcode rank plot and other QC metrics. If the automatic calls are not supported by these diagnostics, consider rerunning cellranger count with --force-cells or applying downstream manual filtering based on UMI counts and other QC measures.
</div>
<!-- We have noticed that sometimes the list of barcodes generated by Cell Ranger exceeds the number expected when performing a sequencing experiment. These needs to be carefully considered (and the list should be filtered) as these cells will tend to be interpreted as non-demultiplexable and might affect overall SNP-IC and cpSNP-IC calculations. -->

For downloading the SNP reference file (~1GB):

In [6]:
url = "https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/data_collections/1000_genomes_project/release/20190312_biallelic_SNV_and_INDEL/ALL.wgs.shapeit2_integrated_snvindels_v2a.GRCh38.27022019.sites.vcf.gz"
save_path = f"{current_directory}/sample_data/ALL.wgs.shapeit2_integrated_snvindels_v2a.GRCh38.27022019.sites.vcf.gz"

response = requests.get(url)
with open(save_path, "wb") as f:
  f.write(response.content)

### 2. Pre-processing (pile-up)

As a pre-processing, we stack all sequencing reads from the BAM file that overlap each genomic position to summarize the bases present at that site. This process is called *pileup*. 

First, we read the sample BAM file using pysam:

In [7]:
import pysam
input_bam = pysam.AlignmentFile(
  f"{current_directory}/sample_data/pbmc_1k_v3_possorted_genome_bam.bam",
  "rb"
)

and display the first few lines

In [8]:
for i,read in enumerate(input_bam.fetch(until_eof=True)):
  print(read)
  if i>=5: break
input_bam.close()

A00228:279:HFWFVDMXX:1:1201:20754:18396	272	1	11274	0	91M	*	0	0	TGGTGGCCAGCGCCCCCTGCTGGCGCCGGGGCACTGCAGGGCCCTCTTGCTTACTGTATAGTGGTGGCACGCCGCCTGCTGGCAGCTAGGG	array('B', [37, 25, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 11, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 11, 37, 37, 25, 37, 25, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 25, 37, 37, 37, 37, 37, 37, 37, 37, 25, 37])	[('NH', 5), ('HI', 4), ('AS', 89), ('nM', 0), ('RE', 'I'), ('li', 0), ('BC', 'CAGCCCAC'), ('QT', 'FFFFFFF,'), ('CR', 'ATTCGTTGTGTAGCAG'), ('CY', 'FFFFF:FFFFFF:FFF'), ('CB', 'ATTCGTTGTGTAGCAG-1'), ('UR', 'CCGTTCCGTTGG'), ('UY', 'F:F:FF:FFF:F'), ('UB', 'CCGTTCCGTTGG'), ('RG', 'pbmc_1k_v3:0:1:HFWFVDMXX:1')]
A00228:279:HFWFVDMXX:1:2258:21395:15280	256	1	11317	0	91M	*	0	0	CTCTTGCTTACTGTATAGTGGTGGCACGCCGCCTGCTGGCAGCTAGGGACATTGCAGGGTCCTCTTGCTCAAGGTGTAGTGGCAGCACGCC	array

We also display the SNP reference

In [9]:
import pandas as pd
input_vcf = pd.read_table(
  f"{current_directory}/sample_data/ALL.wgs.shapeit2_integrated_snvindels_v2a.GRCh38.27022019.sites.vcf.gz", 
  comment="#", 
  header=None,
  names=['chrom','pos','id','ref','alt','qual','filter','info']
)
input_vcf

/tmp/ipykernel_346233/3038289099.py:2: DtypeWarning: Columns (0: chrom) have mixed types. Specify dtype option on import or set low_memory=False.
  input_vcf = pd.read_table(


,chrom,pos,id,ref,alt,qual,filter,info
0,1,10416,.,CCCTAA,C,.,PASS,AC=240;AN=5096;DP=365460;AF=0.05;EAS_AF=0.06;E...
1,1,16103,.,T,G,.,PASS,AC=118;AN=5096;DP=29994;AF=0.02;EAS_AF=0;EUR_A...
2,1,17496,.,AC,A,.,PASS,AC=25;AN=5096;DP=189765;AF=0;EAS_AF=0;EUR_AF=0...
3,1,51479,.,T,A,.,PASS,AC=531;AN=5096;DP=17461;AF=0.1;EAS_AF=0;EUR_AF...
4,1,51898,.,C,A,.,PASS,AC=426;AN=5096;DP=15331;AF=0.08;EAS_AF=0.05;EU...
...,...,...,...,...,...,...,...,...
78229213,X,156029373,.,T,C,.,PASS,AC=181;AN=5096;DP=3661;AF=0.04;EAS_AF=0.05;EUR...
78229214,X,156029383,.,C,G,.,PASS,AC=288;AN=5096;DP=4109;AF=0.06;EAS_AF=0.05;EUR...
78229215,X,156030556,.,A,AG,.,PASS,AC=4;AN=5096;DP=428827;AF=0;EAS_AF=0;EUR_AF=0;...
78229216,X,156030574,.,A,AG,.,PASS,AC=5;AN=5096;DP=462796;AF=0;EAS_AF=0;EUR_AF=0;...


Finally, we calculate how many times each SNP in the reference is associated with each cell in our input BAM file using *cellsnp-lite* with the following command. This process takes about 45 min when using 50 cores. You may skip this step and proceed directly to the visualization stage, as the results are already provided in the output directory.

In [10]:
import oddSNP.utils as utils

utils.call_cslite_pileup(
  bam=f"{current_directory}/sample_data/pbmc_1k_v3_possorted_genome_bam.bam", # bam file. Change here if the down-sampling percentage is modified.
  reference=f"{current_directory}/sample_data/ALL.wgs.shapeit2_integrated_snvindels_v2a.GRCh38.27022019.sites.vcf.gz", # reference file
  barcodes=f"{current_directory}/sample_data/filtered_feature_bc_matrix/barcodes.tsv.gz", # barcode file in the matrix folder
  oupath=f"{current_directory}/sample_data/full_pileup", # output directory
  celltag="CB", # a tag for cell barcode
  umitag="UB",  # a tag for UMI 
  mincount=20,  # minimum number of reads required to report an allele count for a SNP in a cell.
  nproc=50,     # number of cores to use
  force=True    # whether to overwrite
)

cellsnp-lite found: cellsnp-lite 1.2.3 (htslib 1.22.1)

Index file for /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/pbmc_1k_v3_possorted_genome_bam.bam not found. Generating index.


[I::main] start time: 2026-04-17 14:43:08
[W::check_args] Max depth set to maximum value (2147483647)
[W::check_args] Max pileup set to maximum value (2147483647)
[I::main] global settings after checking:
	num of input files = 1
	out_dir = /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/full_pileup
	is_out_zip = 1, is_genotype = 0
	is_target = 0, num_of_pos = 0
	num_of_barcodes = 1222, num_of_samples = 0
	refseq file = (null)
	0 chroms: 
	cell-tag = CB, umi-tag = UB
	nthreads = 50, tp_max_open = 1048576
	mthreads = 50, tp_errno = 0, tp_ntry = 0
	min_count = 20, min_maf = 0.10, doublet_gl = 0
	min_len = 30, min_mapq = 20
	rflag_filter = 772, rflag_require = 0
	max_depth = 2147483647, max_pileup = 2147483647, no_orphan = 1
[I::main] loading the VCF file for given SNPs ...
[I::main] fetching 73257632 candidate variants ...
[I::main] mode 1a: fetch given SNPs in 1222 single cells.
[I::csp_fetch_core][Thread-26] 2.00% SNPs processed.
[I::csp_fetch_core][Thread-46] 2.00% SNPs processed.

Pileup results (generated using cellsnp-lite) for this sample dataset are stored in folder `path/to/oddsnp/sample_data/full_pileup`.

### 3. snpIC Calculation

Using the generated pile-up results, we here compute SNP-IC. As detailed in the manuscript, SNP-IC is a metric for evaluating the performance of with-genotype demultiplexing algorithms and is defined as the sum of UMI counts across each identified SNPs. 

We first load the pile-up results and compute SNP-IC

In [11]:
import pandas as pd 
import oddSNP.snpic as snpic

out = snpic.call_calculate_snpic(
  f"{current_directory}/sample_data/full_pileup", # input directory (pileup results)
  f"{current_directory}/sample_data/full_pileup", # output directory for SNP-IC results
  force=True # whether to overwrite
)
df = pd.read_pickle(out)
print(df)

Saving aggregated pileup to /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/full_pileup/aggregated_pileup.pkl.gz
Saving SNP-IC counts to: /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/full_pileup/snpic_counts.pkl.gz
                       AD     DP   OTH  DPOTH
bcode                                        
AAACCCAAGGAGAGTA-1  365.0  834.0   4.0  838.0
AAACGCTTCAGCCCAG-1  360.0  723.0  10.0  733.0
AAAGAACAGACGACTG-1  222.0  477.0   9.0  486.0
AAAGAACCAATGGCAG-1  170.0  316.0   7.0  323.0
AAAGAACGTCTGCAAT-1  275.0  663.0  10.0  673.0
...                   ...    ...   ...    ...
TTTCCTCTCTCTTGCG-1  458.0  959.0  13.0  972.0
TTTGATCTCTTTGGAG-1  235.0  559.0   8.0  567.0
TTTGGTTAGTAACCTC-1  168.0  355.0   4.0  359.0
TTTGGTTGTAGAATAC-1  425.0  923.0   7.0  930.0
TTTGTTGCAATTAGGA-1  196.0  432.0  11.0  443.0

[1211 rows x 4 columns]


Then we generate the corresponding histogram (x-axis: SNP-IC and y-axis: histogram)

In [12]:
# generate the histogram values
import oddSNP.plotting as plt

hist = snpic.call_generate_histogram(
  f"{current_directory}/sample_data/full_pileup/snpic_counts.pkl.gz", # input SNP-IC results
  f"{current_directory}/sample_data/full_pileup", # output directory
  force=True # whether to overwrite
)

fig = plt.generate_snpic_plot(hist)
fig.show()

Loading SNP-IC counts from: /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/full_pileup/snpic_counts.pkl.gz
Saving SNP-IC histogram to: /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/full_pileup/snpic_histogram.pkl


Alternatively, we can save the plot (as an interactive html file)

In [13]:
# snpic.call_save_snpic_plot(
#   hist,
#   f"{current_directory}/sample_data/full_pileup/snpic_histogram.html",
#   force=True
# )

The histogram suggests that most cells have an SNP-IC greater than the threshold value (50, incidated by the vertical dashed line), so with-genotype demultiplexing algorithms should work on this dataset.

### 4. cpSNP-IC calculation

Using the generated pile-up results, we can also compute cpSNP-IC, which is a metric for evaluating the performance of genotype-free demultiplexing algorithms. As detailed in the manuscript, cpSNP-IC is computed for each cell pair by summing, over all SNPs shared by the pair, the smaller of the two UMI counts for each SNP.

Because this value is computed for cell pairs, it requires more time than the SNP-IC calculation. In the command below, cpSNP-IC is computed for cell pairs with meaningful SNP overlap. This step takes approximately 6 minutes using 15 cores. 

In [14]:
import pandas as pd 
import oddSNP.cpsnpic as cpsnpic

out = cpsnpic.call_calculate_cpsnpic(
  inpath=f"{current_directory}/sample_data/full_pileup/", # input directory
  oupath=f"{current_directory}/sample_data/full_pileup",  # output directory
  nproc=15,           # number of cores to use
  batch_size=10000,   # number of cell pairs to process in each batch
  force=True          # whether to overwrite
  )

df = pd.read_pickle(out)
display(df)

Saving aggregated pileup to /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/full_pileup/aggregated_pileup.pkl.gz
Saving SNPs per cell to /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/full_pileup/snps_per_cell.pkl.gz
Generating cell pairs for 1211 cells
total pairs 732655, total batches 73


Batch progress: 100%|██████████| 74/74 [07:55<00:00,  6.43s/it]


Saving results to /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/full_pileup/cpsnpic_counts.pkl.gz


,cell1,cell2,min_sum
0,TTACGTTTCTCGCTTG-1,TTGGGCGGTCGGAAAC-1,88.0
1,TTACGTTTCTCGCTTG-1,TTGGGTAGTGCTAGCC-1,196.0
2,TTACGTTTCTCGCTTG-1,TTGGGTATCACCGACG-1,78.0
3,TTACGTTTCTCGCTTG-1,TTGGTTTCACTGGATT-1,181.0
4,TTACGTTTCTCGCTTG-1,TTGTGGATCTAAGAAG-1,186.0
...,...,...,...
2522,TTTGATCTCTTTGGAG-1,TTTGGTTGTAGAATAC-1,140.0
2523,TTTGATCTCTTTGGAG-1,TTTGTTGCAATTAGGA-1,91.0
2524,TTTGGTTAGTAACCTC-1,TTTGGTTGTAGAATAC-1,97.0
2525,TTTGGTTAGTAACCTC-1,TTTGTTGCAATTAGGA-1,79.0


We then generate and plot the histogram based on the cpSNP-IC values.

In [15]:
import oddSNP.cpsnpic as cpsnpic
import oddSNP.plotting as plt

# generate the histogram values
hist = cpsnpic.call_generate_histogram(
  '{}/sample_data/full_pileup/cpsnpic_counts.pkl.gz'.format(current_directory), # input directory
  '{}/sample_data/full_pileup'.format(current_directory), # output directory
  force=True # whether to overwrite
)

# plot the histogram
fig = plt.generate_cpsnpic_plot(hist)
fig.show()

Loading cpSNP-IC counts from: /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/full_pileup/cpsnpic_counts.pkl.gz
Saving cpSNP-IC histogram to: /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/full_pileup/cpsnpic_histogram.pkl


Alternatively, we can save the plot (as an html interactive file)

In [16]:
# cpsnpic.call_save_cpsnpic_plot(
#   hist,
#   f"{current_directory}/sample_data/full_pileup/cpsnpic_histogram.html",
#   force=True
# )

The histogram shows that all cells have cpSNP-IC values below the threshold (3,000 indicated by the vertical dashed line), suggesting that genotype-free algorithms will likely perform poorly on this dataset.<u/>

### 5. Simulations

Using the dataset above, we will perform the following three types of simulations with oddSNP:
* A. Sequencing depth and cell number down-sampling. 
* B. SNP reference specification. 
* C. Number of donors.

These simulations allow us to explore the most stringent experimental conditions required for successful demultiplexing. 

#### A. Sequencing depth down-sampling
Using a function in oddSNP, we will downsample the original BAM file. There are two ways to downsample BAM files to reduce their size:
* By number of reads (`reads_downsampling`) - we use the `reads` parameter to determine the percentage of reads from the input file that will be retained. Retained reads are randomly selected and saved to a new file in the selected output folder. The name of the new file (same as original appended with the selected fraction of reads) is returned by the function. This method also automatically generates a list of barcodes included in the output file.
* By number of cells (`barcode_downsampling`)- here we use the `barcodes` parameter to define a list of barcodes to be kept. Retained reads are saved to a new file in the selected output folder. The name of the new file (same as the original appended with the word `filtered`) is returned by the function.

Here, we downsample the sample BAM file to include only 10% of the original reads (reducing the file size 4.5Gb to approximately \~550Mb) using the following command:

In [ ]:
import oddSNP.downsampling as downsampling
import oddSNP.utils as utils

tgt = downsampling.call_reads_downsampling(
  bam=f"{current_directory}/sample_data/pbmc_1k_v3_possorted_genome_bam.bam", # input bam file
  reads=0.1, # fraction of the original reads retained
  oupath=f"{current_directory}/sample_data", # output directory
  celltag="CB", # a tag for cell barcode
  seed=42, # random seed
  nproc=30, # number of cores to use
  force=True # whether to overwrite
)

print(f"Filtered file saved to: {tgt}")

Total number of reads in /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/pbmc_1k_v3_possorted_genome_bam.bam: 76,920,923


Filtering reads: 100%|██████████| 76920923/76920923 [02:14<00:00, 573547.23it/s]


Filtered file saved to: /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/pbmc_1k_v3_possorted_genome_bam_0.10.bam


#### B. SNP reference specification

It is also possible to specify the reference VCF file used for pile-up by using a filtering list of SNP positions. Here, as an example, we will use the manifest file corresponding to Illumina's Infinium Global Diversity Array v1.0 D2 (CSV Format—GRCh38).

First, download the manifest file (200 MB):

In [18]:
import zipfile 

# manifest file
url = "https://webdata.illumina.com/downloads/productfiles/global-diversity-array/infinium-global-diversity-array-8-v1-0-D2-manifest-file-csv.zip"
save_path = f"{current_directory}/sample_data/GDA-8v1-0_D2-csv.zip"
response = requests.get(url)
with open(save_path, "wb") as f:
  f.write(response.content)

# unzip the manifest file
with zipfile.ZipFile(save_path, 'r') as file:
  file.extractall(f"{current_directory}/sample_data/") 
  file.close()

We extract the location of the variants included in the manifest file (chr & position) and store them in a separate file (regions.tsv)

In [19]:
# read initial manifest file
gda = pd.read_csv(f'{current_directory}/sample_data/GDA-8v1-0_D2.csv', 
  header=7, # on inspection, we identified that the actual data starts from row 8 (0-indexed)
  low_memory=False
  )

# save the Chr and MapInfo columns as regions file
gda.dropna(inplace=True)
gda.MapInfo = gda.MapInfo.astype(int)
gda[['Chr', 'MapInfo']
  ].dropna(
  ).to_csv('sample_data/regions.tsv',  # output region file that includes chr & position
    sep='\t', 
    index=False, 
    header=False
  )

Using the bcftools wrapper available in oddSNP, the list of positions is applied to filter the original VCF reference. The resulting reference will contain only the variants found in the intersection of the two datasets.

In [20]:
import oddSNP.downsampling as downsampling
import oddSNP.utils as utils
from importlib import reload
reload(downsampling)
reload(utils)

tgt = downsampling.call_vcf_downsampling(
  vcf=f"{current_directory}/sample_data/ALL.wgs.shapeit2_integrated_snvindels_v2a.GRCh38.27022019.sites.vcf.gz", # input vcf file
  regions=f"{current_directory}/sample_data/regions.tsv", # file containing variant chr&position that we would like to extract
  out=f"{current_directory}/sample_data/filtered_regions.vcf.gz", # output directory
  nproc=30, # number of cores to use
  force=True # whether to overwrite
)

print(f"Filtered VCF saved to: {tgt}")

Index file for /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/ALL.wgs.shapeit2_integrated_snvindels_v2a.GRCh38.27022019.sites.vcf.gz not found. Generating index.
Filtered VCF saved to: /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/filtered_regions.vcf.gz


Using the generated simulated BAM file and SNP reference file, we compute SNP-IC to assess whether these conditions are sufficient for demultiplexing, starting with the pileup calculation.

Notice that, by filtering both the input BAM and the number of variants in the reference, we can greatly reduce the time required for pileup calculation.

In [5]:
import oddSNP.utils as utils

utils.call_cslite_pileup(
  bam=f"{current_directory}/sample_data/pbmc_1k_v3_possorted_genome_bam_0.10.bam", # downsampled bam file. Change here if the down-sampling percentage is modified.
  reference=f"{current_directory}/sample_data/filtered_regions.vcf.gz", # filtered reference file
  barcodes=f"{current_directory}/sample_data/filtered_feature_bc_matrix/barcodes.tsv.gz", # barcode file in the matrix folder
  oupath=f"{current_directory}/sample_data/filtered_pileup", # output directory
  celltag="CB", # a tag for cell barcode
  umitag="UB",  # a tag for UMI 
  mincount=20,  # minimum number of reads required to report an allele count for a SNP in a cell.
  nproc=50,     # number of cores to use
  force=True    # whether to overwrite
)

cellsnp-lite found: cellsnp-lite 1.2.3 (htslib 1.22.1)



[I::main] start time: 2026-04-20 10:02:36
[W::check_args] Max depth set to maximum value (2147483647)
[W::check_args] Max pileup set to maximum value (2147483647)
[I::main] global settings after checking:
	num of input files = 1
	out_dir = /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/filtered_pileup
	is_out_zip = 1, is_genotype = 0
	is_target = 0, num_of_pos = 0
	num_of_barcodes = 1222, num_of_samples = 0
	refseq file = (null)
	0 chroms: 
	cell-tag = CB, umi-tag = UB
	nthreads = 50, tp_max_open = 1048576
	mthreads = 50, tp_errno = 0, tp_ntry = 0
	min_count = 20, min_maf = 0.10, doublet_gl = 0
	min_len = 30, min_mapq = 20
	rflag_filter = 772, rflag_require = 0
	max_depth = 2147483647, max_pileup = 2147483647, no_orphan = 1
[I::main] loading the VCF file for given SNPs ...
[I::main] fetching 170318 candidate variants ...
[I::main] mode 1a: fetch given SNPs in 1222 single cells.
[I::csp_fetch_core][Thread-1] 2.03% SNPs processed.
[I::csp_fetch_core][Thread-1] 4.02% SNPs processed.

Pileup results for the downsampled dataset are stored in folder `path/to/oddsnp/sample_data/filtered_pileup`.

We can repeat the process (SNP-IC calculation and histogram plotting) for the pileup results obtained using only 10% of the original reads and the GDA SNP reference.

In [22]:
# 10% BAM
# Reduced Reference
import pandas as pd 
import oddSNP.snpic as snpic 
import oddSNP.plotting as plt

out = snpic.call_calculate_snpic(
  f"{current_directory}/sample_data/filtered_pileup", # input directory including pileup results
  f"{current_directory}/sample_data/filtered_pileup", # output directory
  force=True # whether to overwrite
)
df = pd.read_pickle(out)
print(df)

# generate the histogram values
hist = snpic.call_generate_histogram(
  f"{current_directory}/sample_data/filtered_pileup/snpic_counts.pkl.gz", # input SNP-IC result file
  f"{current_directory}/sample_data/filtered_pileup", # output directory
  force=True # whether to overwrite
)

fig = plt.generate_snpic_plot(hist)
fig.show()

# alternatively, we can save the plot (as an html interactive file)
# snpic.call_save_snpic_plot(
#   hist,
#   f"{current_directory}/sample_data/filtered_pileup/snpic_histogram.html",
#   force=True # whether to overwrite
# )

Saving aggregated pileup to /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/filtered_pileup/aggregated_pileup.pkl.gz
Saving SNP-IC counts to: /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/filtered_pileup/snpic_counts.pkl.gz
                     AD   DP  OTH  DPOTH
bcode                                   
AAACCCAAGGAGAGTA-1  3.0  9.0  0.0    9.0
AAACGCTTCAGCCCAG-1  3.0  8.0  0.0    8.0
AAAGAACAGACGACTG-1  1.0  1.0  0.0    1.0
AAAGAACCAATGGCAG-1  1.0  2.0  0.0    2.0
AAAGAACGTCTGCAAT-1  2.0  2.0  0.0    2.0
...                 ...  ...  ...    ...
TTTCCTCTCTCTTGCG-1  0.0  2.0  0.0    2.0
TTTGATCTCTTTGGAG-1  2.0  3.0  0.0    3.0
TTTGGTTAGTAACCTC-1  1.0  2.0  0.0    2.0
TTTGGTTGTAGAATAC-1  2.0  3.0  0.0    3.0
TTTGTTGCAATTAGGA-1  0.0  1.0  0.0    1.0

[951 rows x 4 columns]
Loading SNP-IC counts from: /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/filtered_pileup/snpic_counts.pkl.gz
Saving SNP-IC histogram to: /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/filtered_

Our criterion SNP-IC=50 is also plotted as a vertial line. <u>In this case, 10%-downsampling and the use of the GDA SNP reference are not good choices, as many cells have SNP-IC values lower than 50. 

By chaning the downsampling percentage, one can explore the optimal sequencing depth for demultiplexing.</u> 

#### C. Number of donor simulations

Here, we estimate the distribution of cells that can be succesfully demultiplexed, given a specified number of donors. As input, we provide either a list representing the targeted donor ratios, or simply the number of donors (notice in both cases we use the `n` argument):

In [23]:
# number of donors 
n=5
# or list of probabilities (must add up to 1) 
# n=[0.1,0.3,0.4,0.2]

(If the number of donors is provided (i.e., n is an integer), we assume that cells from each donor are equally distributed.)

We then assume that the actual donor distributions are determined through random sampling (following a multinomial distribution). Based on this and the SNP-IC threshold, we estimate the percentage of cells that can be correctly assigned to their respective donors. 

In [24]:
import oddSNP.snpic as snpic
import oddSNP.plotting as plt

out = snpic.call_estimate_donor_assignment(
  barfile=f"{current_directory}/sample_data/filtered_feature_bc_matrix/barcodes.tsv.gz", # cell barcode file
  agfile=f"{current_directory}/sample_data/filtered_pileup/snpic_counts.pkl.gz", # SNP-IC results
  oupath=f"{current_directory}/sample_data/filtered_pileup/", # output directory
  n=n, # we either provide number of donors or a list of probabilities
  seed=42, # random seed
  force=True # whether to overwrite
)

df = pd.read_pickle(out)
print(df)

fig = plt.generate_donor_distribution_plot(out)
fig.show()

Using --n equally distributed donors for barcode/donor mapping estimation
[221 243 244 261 253]
Saving donor estimation bars to: /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/filtered_pileup/snpic_bars.pkl
                    donor  correct
bcode                             
AAACCCAAGGAGAGTA-1      0    False
AAACGCTTCAGCCCAG-1      1    False
AAAGAACAGACGACTG-1      0    False
AAAGAACCAATGGCAG-1      3    False
AAAGAACGTCTGCAAT-1      2    False
...                   ...      ...
TTTCCTCTCTCTTGCG-1      3    False
TTTGATCTCTTTGGAG-1      2    False
TTTGGTTAGTAACCTC-1      2    False
TTTGGTTGTAGAATAC-1      4    False
TTTGTTGCAATTAGGA-1      3    False

[951 rows x 2 columns]


In this case, most cells are unassigned because they have low SNP-IC values, as shown in the previously obtained histogram.

Similarly, if we perform the same simulations using the original pile-up results, we obtain:

In [25]:
import oddSNP.snpic as snpic
import oddSNP.plotting as plt

out = snpic.call_estimate_donor_assignment(
  barfile=f"{current_directory}/sample_data/filtered_feature_bc_matrix/barcodes.tsv.gz", # cell barcode file
  agfile=f"{current_directory}/sample_data/full_pileup/snpic_counts.pkl.gz", # SNP-IC results
  oupath=f"{current_directory}/sample_data/full_pileup/", # output directory
  n=n, # we either provide number of donors or a list of probabilities
  seed=42, # random seed
  force=True # whether to overwrite
)

df = pd.read_pickle(out)
print(df)

fig = plt.generate_donor_distribution_plot(out)
fig.show()

Using --n equally distributed donors for barcode/donor mapping estimation
[221 243 244 261 253]
Saving donor estimation bars to: /mnt/storage4/rallendes/oddSNP/notebooks/sample_data/full_pileup/snpic_bars.pkl
                    donor  correct
bcode                             
AAACCCAAGGAGAGTA-1      0     True
AAACGCTTCAGCCCAG-1      1     True
AAAGAACAGACGACTG-1      0     True
AAAGAACCAATGGCAG-1      3     True
AAAGAACGTCTGCAAT-1      2     True
...                   ...      ...
TTTCCTCTCTCTTGCG-1      3     True
TTTGATCTCTTTGGAG-1      2     True
TTTGGTTAGTAACCTC-1      2     True
TTTGGTTGTAGAATAC-1      4     True
TTTGTTGCAATTAGGA-1      3     True

[1211 rows x 2 columns]


Most cells are correctly assigned to their respective donors. 